# 05 · Per-tile attributes

**Purpose:** Enrich tiles with reference building density and size attributes.

**Inputs:** `outputs/metrics/` tile parquets, AOI reference data

**Outputs:** `outputs/scratch/per_tile_enriched_all_cities.csv`

**Run order:** After `04`.

**Last run:** _(fill in when you run it)_

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 07 — Per-tile building attributes (enrichment)

Enriches existing per-tile validation outputs with reference building
density and average size, computed via centroid-based assignment.

**Non-destructive:** the original `vector_metrics_tiles_all_datasets.parquet`
is never modified. Results are written to:
- `outputs/metrics/<city>/vector_metrics_tiles_enriched.parquet` — per city
- `outputs/scratch/per_tile_enriched_all_cities.csv` — combined flat CSV

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/urban_validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg['data_dir']
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
OUT_SCRATCH  = PROJECT_ROOT / 'outputs' / 'scratch'
OUT_SCRATCH.mkdir(parents=True, exist_ok=True)

SENTINEL = 'vector_metrics_tiles_all_datasets.parquet'
ENRICHED = 'vector_metrics_tiles_enriched.parquet'

# Candidate dataset names from config — used to EXCLUDE candidate parquets when
# discovering reference files via glob. Any file in {city}/vector/ that does NOT
# start with {city_underscore}_{cand_name} is treated as a reference file.
CAND_NAMES = [
    d['name']
    for d in cfg.get('vector', {}).get('datasets', [])
    if d.get('enabled', True)
]


def find_ref_files(city_slug: str) -> list:
    """Discover reference building files for a city by excluding known candidate files.

    Candidate files follow the pipeline pattern: {city_underscored}_{dataset_name}*.parquet
    Everything else in the vector/ directory is treated as a reference file.
    This is robust to any reference file naming convention (WBG, HOT-OSM, SpaceNet7, etc.)
    and does not depend on the AOI tracker CSV being populated.
    """
    vec_dir = DATA_DIR / city_slug / 'vector'
    if not vec_dir.exists():
        return []
    city_prefix = city_slug.lower().replace('-', '_')
    cand_starts = {f"{city_prefix}_{name}" for name in CAND_NAMES}
    refs = []
    for ext in ('*.geojson', '*.gpkg', '*.shp', '*.parquet'):
        for f in vec_dir.glob(ext):
            if not any(f.stem.lower().startswith(pat) for pat in cand_starts):
                refs.append(f)
    return sorted(refs)


print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'METRICS_ROOT  : {METRICS_ROOT}  exists={METRICS_ROOT.exists()}')
print(f'Candidate datasets to exclude : {CAND_NAMES}')

# Quick sanity check on known problem cities from prior run
for test_city in ['afg-ayabak', 'ant-curacao']:
    refs = find_ref_files(test_city)
    print(f'  {test_city}: {[f.name for f in refs]}')


In [ ]:
def load_tiles(city_slug: str, data_dir: Path) -> gpd.GeoDataFrame | None:
    """Load tile grid GPKG written by BaseValidationRunner._save_tiles()."""
    tiles_path = data_dir / city_slug / 'tiles' / f'{city_slug.lower()}_tiles.gpkg'
    if not tiles_path.exists():
        return None
    return gpd.read_file(tiles_path)


def load_reference(ref_paths: list) -> gpd.GeoDataFrame | None:
    """Load and merge reference footprints from a list of Paths.

    All files are reprojected to the CRS of the first successfully loaded file
    before concatenation, so mixed EPSG:4326 / EPSG:3857 datasets are handled.
    """
    gdfs = []
    for p in ref_paths:
        if not p.exists():
            print(f'  [warn] ref file not found: {p}')
            continue
        try:
            gdf = gpd.read_parquet(p) if p.suffix.lower() == '.parquet' else gpd.read_file(p)
            gdfs.append(gdf)
        except Exception as e:
            print(f'  [warn] could not read {p.name}: {e}')

    if not gdfs:
        return None
    if len(gdfs) == 1:
        return gdfs[0]

    # Normalize to the CRS of the first file before concat
    target_crs = gdfs[0].crs
    aligned = [gdfs[0]]
    for gdf in gdfs[1:]:
        if gdf.crs != target_crs:
            gdf = gdf.to_crs(target_crs)
        aligned.append(gdf)
    return gpd.GeoDataFrame(pd.concat(aligned, ignore_index=True), crs=target_crs)


def enrich_tile_metrics(tile_metrics_df: pd.DataFrame,
                        tiles_gdf: gpd.GeoDataFrame,
                        ref_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Add 4 centroid-based building attribute columns to tile_metrics_df."""
    if ref_gdf.crs != tiles_gdf.crs:
        ref_gdf = ref_gdf.to_crs(tiles_gdf.crs)

    ref_pts = ref_gdf.copy()
    ref_pts['area_m2'] = ref_gdf.geometry.area
    ref_pts['geometry'] = ref_gdf.geometry.centroid

    joined = gpd.sjoin(
        ref_pts[['geometry', 'area_m2']],
        tiles_gdf[['tile_id', 'geometry']],
        how='inner',
        predicate='within',
    )

    per_tile = joined.groupby('tile_id').agg(
        ref_building_count_centroid=('area_m2', 'count'),
        mean_ref_building_area_m2=('area_m2', 'mean'),
    ).reset_index()

    tiles_area = tiles_gdf[['tile_id', 'geometry']].copy()
    tiles_area['tile_area_km2'] = tiles_area.geometry.area / 1e6

    merged = tiles_area[['tile_id', 'tile_area_km2']].merge(per_tile, on='tile_id', how='left')
    merged['ref_building_count_centroid'] = (
        merged['ref_building_count_centroid'].fillna(0).astype(int)
    )
    merged['ref_building_density_per_km2'] = (
        merged['ref_building_count_centroid']
        / merged['tile_area_km2'].replace(0, np.nan)
    )

    return tile_metrics_df.merge(
        merged.drop(columns='geometry', errors='ignore'),
        on='tile_id',
        how='left',
    )

print('Helper functions defined.')


In [ ]:
NEW_COLS = ['tile_area_km2', 'ref_building_count_centroid',
            'mean_ref_building_area_m2', 'ref_building_density_per_km2']

city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(SENTINEL))
print(f'Found {len(city_dirs)} processed cities\n')

all_enriched = []
skipped = []

for city_dir in city_dirs:
    city = city_dir.name
    sentinel_path = city_dir / SENTINEL
    enriched_path = city_dir / ENRICHED

    try:
        tile_df = pd.read_parquet(sentinel_path)
    except Exception as e:
        print(f'[SKIP] {city}: cannot read parquet — {e}')
        skipped.append(city)
        continue

    if all(c in tile_df.columns for c in NEW_COLS):
        print(f'[SKIP] {city}: new columns already present')
        all_enriched.append(tile_df)
        continue

    tiles = load_tiles(city, DATA_DIR)
    ref_paths = find_ref_files(city)

    # ── Full centroid-based enrichment (ref files on disk) ────────────────────
    if tiles is not None and ref_paths:
        ref = load_reference(ref_paths)
        if ref is not None:
            try:
                enriched_df = enrich_tile_metrics(tile_df, tiles, ref)
                enriched_df.to_parquet(enriched_path, index=False)
                mean_density = enriched_df['ref_building_density_per_km2'].mean()
                ref_names = ', '.join(p.name for p in ref_paths)
                print(f'[OK]   {city}: {len(enriched_df)} tiles | '
                      f'mean density={mean_density:.1f} bldg/km² | ref={ref_names}')
                all_enriched.append(enriched_df)
                continue
            except Exception as e:
                print(f'[ERR]  {city}: centroid enrichment failed — {e}')

    # ── Fallback: derive from existing parquet columns ─────────────────────────
    # Used when ref files are gone from disk (data cleaned up post-pipeline run).
    if tiles is not None and 'n_ref' in tile_df.columns and 'ref_area_total_m2' in tile_df.columns:
        try:
            tile_areas = tiles[['tile_id', 'geometry']].copy()
            tile_areas['tile_area_km2'] = tile_areas.geometry.area / 1e6
            tile_area_map = tile_areas.set_index('tile_id')['tile_area_km2']

            first_ds = tile_df['dataset'].iloc[0] if 'dataset' in tile_df.columns else None
            ref_per_tile = (
                tile_df[tile_df['dataset'] == first_ds][['tile_id', 'n_ref', 'ref_area_total_m2']]
                if first_ds else tile_df[['tile_id', 'n_ref', 'ref_area_total_m2']]
            ).drop_duplicates('tile_id')

            enriched_df = tile_df.copy()
            enriched_df['tile_area_km2'] = enriched_df['tile_id'].map(tile_area_map)
            n_ref_map = ref_per_tile.set_index('tile_id')['n_ref']
            area_map  = ref_per_tile.set_index('tile_id')['ref_area_total_m2']
            enriched_df['ref_building_count_centroid'] = enriched_df['tile_id'].map(n_ref_map).fillna(0).astype(int)
            enriched_df['mean_ref_building_area_m2']   = (
                enriched_df['tile_id'].map(area_map) / enriched_df['ref_building_count_centroid'].replace(0, np.nan)
            )
            enriched_df['ref_building_density_per_km2'] = (
                enriched_df['ref_building_count_centroid'] / enriched_df['tile_area_km2'].replace(0, np.nan)
            )
            enriched_df.to_parquet(enriched_path, index=False)
            mean_density = enriched_df['ref_building_density_per_km2'].mean()
            print(f'[APPROX] {city}: {len(enriched_df)} tiles | '
                  f'mean density={mean_density:.1f} bldg/km² [intersection approx]')
            all_enriched.append(enriched_df)
            continue
        except Exception as e:
            print(f'[ERR]  {city}: fallback enrichment failed — {e}')

    reason = 'no tiles GPKG' if tiles is None else 'no ref files in vector/ dir'
    print(f'[SKIP] {city}: {reason}')
    skipped.append(city)

if all_enriched:
    combined = pd.concat(all_enriched, ignore_index=True)
    csv_path = OUT_SCRATCH / 'per_tile_enriched_all_cities.csv'
    combined.to_csv(csv_path, index=False)
    print(f'\nCombined CSV → {csv_path}  ({len(combined)} rows, {combined["city"].nunique()} cities)')
    print(f'Skipped: {len(skipped)} cities: {skipped}')
else:
    print('No cities enriched.')


In [ ]:
if 'combined' not in dir() or combined is None:
    print('Run the enrichment cell first.')
else:
    # (1) Full column list
    print('=== Columns in enriched parquet ===')
    print(combined.columns.tolist())

    # (2) Summary stats per city for the 4 new columns
    print('\n=== Per-city summary stats ===')
    print(combined.groupby('city')[NEW_COLS].agg(['mean', 'std']).round(3).to_string())

    # (3) Scatter: ref_building_density_per_km2 vs f1, coloured by city
    fig, ax = plt.subplots(figsize=(10, 6))
    cities = combined['city'].unique()
    colours = cm.tab20(np.linspace(0, 1, len(cities)))
    for colour, city in zip(colours, sorted(cities)):
        grp = combined[combined['city'] == city]
        ax.scatter(
            grp['ref_building_density_per_km2'], grp['f1'],
            s=8, alpha=0.35, color=colour, label=city,
        )
    ax.set_xlabel('Reference building density (buildings / km\u00b2)')
    ax.set_ylabel('F1 score')
    ax.set_title('Reference building density vs F1 \u2014 all processed cities')
    ax.legend(markerscale=4, fontsize=7, ncol=2, loc='lower right')
    plt.tight_layout()
    scatter_path = OUT_SCRATCH / 'density_vs_f1_all_cities.png'
    plt.savefig(scatter_path, dpi=150)
    plt.show()
    print(f'Scatter saved \u2192 {scatter_path}')

## Notes

- **Non-destructive**: original `vector_metrics_tiles_all_datasets.parquet` is never modified.
- **Centroid-based assignment**: each reference building is assigned to the tile containing
  its centroid. Buildings straddling a tile boundary are counted only in the tile where the
  centroid falls, avoiding double-counting.
- **Future runs**: new cities processed after the pipeline update (commit `0daa7fb`) will
  already have these columns in their sentinel parquet and can be loaded directly.
- **Density units**: buildings per km² using the projected tile area (metres CRS), not
  geographic area — consistent with the pipeline's per-city density summary.